# STGCN Baseline
In this notebook, we instantiate and train the Spatio-Temporal Graph Convolutional Network (STGCN) using our standard pipeline and tracking to TensorBoard.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import datetime
from tools.dataloading import get_motorimagery_loaders
from tools.preprocessing import apply_epoch_ica
from tools.training import Trainer
from tools.graph import build_edge_index
from models.stgcn.model import STGCN
import numpy as np

In [8]:
# 1. Pipeline Configuration
from moabb.datasets import BNCI2014_001, Gao2026

dataset = Gao2026()

hyperparams = {
    "dataset": type(dataset).__name__,
    "split_mode": "loso",
    "test_subject_id": 1,
    "batch_size": 32,
    "sample_frequency": 128,
    "learning_rate": 0.001,
    "epochs": 300,
    "patience": 50,
    "K": 3,
    "preprocessing": "ICA"
}

# Add dynamic timestamp string to organize logs
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = f"models/stgcn/results/baseline_{hyperparams['dataset']}_{hyperparams['split_mode']}_{timestamp}"

# 2. Data Loading
print(
    f"Loading {hyperparams['dataset']} with {hyperparams['split_mode'].upper()} split for Subject {hyperparams['test_subject_id']}...")

train_loader, test_loader, metadata = get_motorimagery_loaders(
    dataset,
    preprocessing_fn=apply_epoch_ica,
    test_subject_id=hyperparams["test_subject_id"],
    sample_frequency=hyperparams["sample_frequency"],
    batch_size=hyperparams["batch_size"],
    split_mode=hyperparams["split_mode"]
)

Choosing from all possible events


Loading Gao2026 with LOSO split for Subject 1...


Missing BDF file: /home/carlos/mne_data/MNE-gao2026-data/sub-08/ses-02/eeg/sub-08_ses-02_task-AVI_eeg.bdf


Applying FastICA (n_components=15) to epoched data...


Split Mode: LOSO | Test Subject: 1
Training on 15840 trials from 21 subjects
Testing on 800 trials from 1 subject
Input Shape for Model: torch.Size([32, 1, 32, 512])
Batch size: 32 | Nº Channels: 32 | Sample length: 512


In [9]:
# 3. Extract channels coordinates to build the edge_index matrix dynamically
# get_data returns a nested dict: data[subject_id][session_id][run_id] -> mne.io.Raw
subject_id = hyperparams["test_subject_id"]
raw_data = dataset.get_data([subject_id])
subject_data = raw_data[subject_id]
first_session = list(subject_data.keys())[0]
first_run = list(subject_data[first_session].keys())[0]
raw = subject_data[first_session][first_run]

info = raw.info

channel_positions = {}
for ch in info['chs']:
    channel_positions[ch['ch_name']] = ch['loc'][:3]  # The 3D coordinates

edge_index, edge_weight = build_edge_index(channel_positions, n_closest=4)
print("edge_index shape:", edge_index.shape)

edge_index shape: torch.Size([2, 128])


In [10]:
# Reconfigure shapes to match the model using the metadata loaded previously
hyperparams["sample_length"] = metadata["sample_length"]
hyperparams["electrode_channels"] = metadata["electrode_channels"]
hyperparams["n_classes"] = metadata["n_classes"]

print(
    f"Time Samples: {hyperparams['sample_length']}, Channels: {hyperparams['electrode_channels']}, Classes: {hyperparams['n_classes']}")

Time Samples: 512, Channels: 32, Classes: 10


In [11]:
# 4. Model Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = STGCN(
    n_channels=hyperparams["electrode_channels"],
    n_classes=hyperparams["n_classes"],
    time_samples=hyperparams["sample_length"],
    K=hyperparams["K"]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["learning_rate"])

Using device: cuda


In [12]:
# 5. Training and Evaluation Tracking

# We also need to move edge index and weight to device
edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    log_dir=log_dir,
    experiment_config=hyperparams
)

# Pass edge_index and edge_weight explicitly to the training loops using kwargs
trainer.train(
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=hyperparams["epochs"],
    patience=hyperparams["patience"],
    edge_index=edge_index,
    edge_weight=edge_weight
)

Starting training on cuda...
Logging TensorBoard to: models/stgcn/results/baseline_Gao2026_loso_20260329-121818
Epoch [1/300] | Train Loss: 2.4211 | Test Loss: 2.3925 | Test Acc: 9.50%
Epoch [10/300] | Train Loss: 2.1738 | Test Loss: 2.6579 | Test Acc: 10.38%
Epoch [20/300] | Train Loss: 2.0141 | Test Loss: 2.9192 | Test Acc: 11.25%
Epoch [30/300] | Train Loss: 1.9068 | Test Loss: 3.2073 | Test Acc: 10.25%
Epoch [40/300] | Train Loss: 1.8371 | Test Loss: 3.2670 | Test Acc: 9.88%
Epoch [50/300] | Train Loss: 1.7677 | Test Loss: 3.7857 | Test Acc: 9.50%
Early stopping triggered after 51 epochs.
Training complete. Best model saved to: models/stgcn/results/baseline_Gao2026_loso_20260329-121818/best_model.pth


In [ ]:
# Evaluate on unseen test set using identical kwargs
test_loss, test_acc = trainer.evaluate(
    test_loader, edge_index=edge_index, edge_weight=edge_weight)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")